# Fábrica de Corpus: Radar Cidadão

Disciplina de NLP, Global Solution 2026.2 (FIAP, CitizenTech). Este notebook transforma documentos públicos sobre chuvas intensas, alagamentos e deslizamentos em três corpus JSONL (`corpus_texto`, `corpus_qa` e `corpus_chat`) e abre uma interface Gradio para testar a ferramenta.

**Como rodar no Google Colab**

1. Abra este notebook no Colab (Arquivo > Abrir notebook > GitHub, ou pelo link do README).
2. O ambiente **sem GPU** basta: a geração com o Qwen é opcional e está desligada (etapa 5).
3. Execute as células em ordem (Ambiente de execução > Executar tudo). A primeira clona o repositório e instala as bibliotecas; leva alguns minutos.
4. Na última célula aparece um link público do Gradio (`share=True`) para usar a interface.

Fluxo da ferramenta: ingestão -> extração com a fonte -> limpeza e normalização -> anonimização -> segmentação (256 a 512 tokens) -> geração com o Qwen (opcional, GPU) -> revisão humana -> finalização e relatório de qualidade.

In [ ]:
# 0. Preparação: no Colab, clona o repositório e instala as bibliotecas; fora dele, usa a pasta local
import subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    REPO = Path("/content/radar-cidadao")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sarahmfurlan-sudo/radar-cidadao.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    PROJETO = REPO / "nlp-fabrica-corpus"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJETO / "requirements.txt")], check=True)
else:
    PROJETO = Path.cwd() if (Path.cwd() / "src" / "fabrica_corpus").exists() else Path.cwd() / "nlp-fabrica-corpus"

if str(PROJETO / "src") not in sys.path:
    sys.path.insert(0, str(PROJETO / "src"))
commit = subprocess.run(["git", "-C", str(PROJETO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Projeto:", PROJETO, "| commit:", commit or "(sem git)", "| Colab:", EM_COLAB)


def mostrar(texto_md):
    """Mostra Markdown formatado no notebook (ou texto puro fora dele)."""
    try:
        from IPython.display import Markdown, display
        display(Markdown(texto_md))
    except ImportError:
        print(texto_md)

## 1 e 2. Ingestão e extração com a fonte

Lê os 5 documentos de `dados/entrada` (PDF, HTML) aplicando a seleção de páginas de `selecao_paginas.json` (capa, créditos, sumário e referências ficam de fora). Cada trecho guarda o arquivo de origem e a página.

In [ ]:
from collections import Counter
from fabrica_corpus.exportacao import ENTRADA
from fabrica_corpus.ingestao import ingerir_pasta

trechos = ingerir_pasta(ENTRADA)
por_fonte = Counter(t.fonte for t in trechos)
print(f"{len(trechos)} trechos extraídos de {len(por_fonte)} documentos:")
for fonte, n in por_fonte.items():
    print(f"  {fonte}: {n} trecho(s)")
t = trechos[0]
print(f"\nExemplo: fonte={t.fonte} | tipo={t.tipo} | página={t.pagina}\n{t.texto[:400]}")

## 3 e 4. Limpeza e normalização

Unicode e aspas, cabeçalhos e rodapés repetidos, números de página, sumário, rodapé de página web, hifenização e letra capitular, linhas quebradas, blocos de referências bibliográficas e espaços. Acentos, pontuação e maiúsculas são mantidos; não há remoção de stop words nem stemming.

In [ ]:
from fabrica_corpus.limpeza import limpar_corpus

limpos, est_limpeza = limpar_corpus(trechos)
print("Caracteres antes:", est_limpeza["caracteres_antes"], "| depois:", est_limpeza["caracteres_depois"])
print("Trechos vazios descartados:", est_limpeza["trechos_vazios_descartados"])
print("Linhas de referência removidas:", est_limpeza["linhas_de_referencia_removidas"])
print("Cabeçalhos/rodapés repetidos por documento:", est_limpeza["repetidas_por_documento"])

## 5. Anonimização

CPF, e-mail, telefone, nomes de pessoas (spaCy `pt_core_news_md` + regras) e sobrenomes de autores em citações viram `[CPF]`, `[EMAIL]`, `[TELEFONE]` e `[NOME]`. Números de emergência (190, 192, 193, 199, 40199) e o 156 são preservados. Abaixo, o "antes e depois" no arquivo de teste com **dados fictícios** e as contagens no corpus real.

In [ ]:
from fabrica_corpus.anonimizacao import anonimizar_corpus
from fabrica_corpus.ingestao import extrair_arquivo

teste, _ = limpar_corpus(extrair_arquivo(PROJETO / "dados" / "teste" / "teste_dados_ficticios.txt"))
teste_anon, cont_teste = anonimizar_corpus(teste)
print("=== ANTES (dados fictícios) ===\n" + teste[0].texto)
print("\n=== DEPOIS ===\n" + teste_anon[0].texto)
print("\nContagem no arquivo de teste:", cont_teste)

anonimos, est_anon = anonimizar_corpus(limpos)
print("Contagem no corpus real (5 documentos):", est_anon)

## 6. Segmentação

Trechos de 256 a 512 tokens, contados com o tokenizador do Qwen 2.5 (o mesmo vocabulário do modelo gerador), sempre cortando em fim de frase. Cada segmento recebe um `segmento_id` (arquivo + número) e guarda as páginas de origem. Na primeira execução o tokenizador é baixado do Hugging Face.

In [ ]:
from fabrica_corpus.segmentacao import MODELO_TOKENIZADOR, segmentar

segmentos, est_seg = segmentar(anonimos)
print("Tokenizador:", MODELO_TOKENIZADOR)
print({k: est_seg[k] for k in ("segmentos", "tokens_media", "tokens_min", "tokens_max", "segmentos_curtos")})
print("Segmentos por documento:", est_seg["segmentos_por_fonte"])
s = segmentos[0]
print(f"\n{s.id} | páginas {s.paginas} | {s.tokens} tokens\n{s.texto[:500]}")

## 7. Geração das perguntas e dos diálogos (opcional, precisa de GPU)

As perguntas (`corpus_qa`) e os diálogos (`corpus_chat`) foram gerados com o **Qwen2.5-14B-Instruct** em 4 bits, no Colab com GPU, a partir dos fatos extraídos de cada segmento; passaram por travas automáticas, por um juiz de coerência e pela revisão humana de 20% da amostra. Os corpus entregues são essa versão **revisada**, em `entrega-final/`.

Esta etapa fica **desligada** (`RODAR_GERACAO = False`): gerar de novo exige GPU, leva horas e produziria exemplos sem revisão. Para demonstrar, ligue a opção num ambiente com GPU; ela gera só alguns exemplos, sem tocar na entrega.

In [ ]:
from fabrica_corpus import geracao

RODAR_GERACAO = False   # mude para True só num ambiente com GPU

print("Modelo:", geracao.MODELO_GERADOR, "| prompt QA/chat:", geracao.VERSAO_QA_CHAT, "| recusas:", geracao.VERSAO_PROMPT)
if RODAR_GERACAO:
    import torch
    assert torch.cuda.is_available(), "Ative uma GPU (Ambiente de execução > Alterar o tipo de ambiente)."
    gerar = geracao.carregar_qwen()
    demo_qa, demo_est = geracao.gerar_qa(segmentos, gerar, total=4, por_segmento=2)
    for e in demo_qa:
        print("\nP:", e["pergunta"], "\nR:", e["resposta"])
else:
    print("Geração desligada: os corpus usados a seguir são os da versão revisada (entrega-final).")

## 8. Finalização e relatório de qualidade

A finalização trabalha sobre os corpus **revisados**: refaz o `corpus_texto` pelo pipeline acima, tira as páginas das citações, valida campos, falas, duplicatas, quantidades mínimas e recusas válidas. Aqui ela roda numa **cópia** da `entrega-final`, só para mostrar que o resultado é reproduzido; depois vem o relatório (quantidades, média de tokens, duplicatas e 5 exemplos aleatórios de cada corpus).

In [ ]:
import json, shutil, tempfile
from fabrica_corpus.finalizacao import ENTREGA, finalizar

copia = Path(tempfile.mkdtemp()) / "entrega"
shutil.copytree(ENTREGA, copia)
metricas = finalizar(copia, segmentos=segmentos)
for nome in ("corpus_texto.jsonl", "corpus_qa.jsonl", "corpus_chat.jsonl"):
    igual = (copia / nome).read_text(encoding="utf-8") == (ENTREGA / nome).read_text(encoding="utf-8").replace("\r\n", "\n")
    print(f"{nome}: reproduzido igual à entrega? {igual}")
print("Recusas válidas:", metricas["recusas_validas"], "| respostas com página citada:", metricas["respostas_com_paginas"])

In [ ]:
from fabrica_corpus.interface import exemplos_corpus, resumo_corpus

mostrar(resumo_corpus(ENTREGA))
mostrar(exemplos_corpus(ENTREGA, semente=2026))

## 9. Interface Gradio

- **Processar documento:** envie PDF, TXT, DOCX ou HTML (vários arquivos) ou cole um texto e veja cada etapa: extração com a fonte, limpeza, anonimização e segmentação. No fim, baixe o `corpus_texto.jsonl` daquele envio. Para testar a anonimização, use `dados/teste/teste_dados_ficticios.docx` (dados fictícios, com tabela de contatos).
- **Corpus finais:** estatísticas, 5 exemplos aleatórios de cada corpus e download dos 3 JSONL da entrega.

No Colab, `share=True` gera um link público temporário.

In [ ]:
from fabrica_corpus.interface import criar_app

app = criar_app(ENTREGA)
if EM_COLAB:
    app.launch(share=True)
else:
    print("Fora do Colab: rode app.launch() para abrir a interface localmente.")